# Bengali Toxicity — v13 (pseudo-labeling / self-training)

**New method, evidence-based.** The bottleneck is data size (~3.8k rows). External data is banned, but the **1000 test texts are in-domain and usable via self-training**:
1. Train base+large on real data (the v10 recipe, our 0.573 winner).
2. Predict the test set; keep only **high-confidence** rows.
3. Retrain on **train + confident test pseudo-labels** (~1.3-1.4x more data).
4. Blend round-1 and round-2, threshold on real OOF, submit.

Uses only competition data. Official mount, test.csv IDs. ~3-4h. **GPU + Internet ON, Run All.**


In [ ]:
# ---- 1. Install ----
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","iterative-stratification==0.1.7"], check=False)
print("deps done")

In [ ]:
# ---- 2. Imports & CONFIG ----
import os, re, gc, glob, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, logging as hf_logging
from torch.optim import AdamW
hf_logging.set_verbosity_error()
MODELS=[
    dict(name="csebuetnlp/banglabert",       folds=5, batch=16, accum=1, lr=2e-5,   epochs=4),
    dict(name="csebuetnlp/banglabert_large", folds=3, batch=8,  accum=2, lr=1.5e-5, epochs=4),
]
class CFG:
    max_len=128; lstm_hidden=256; dropout=0.3; weight_decay=0.01; warmup_ratio=0.1
    pos_weight_clip=30.0; GATE=True; seed=42; use_fp16=True
    conf_hi=0.90; conf_lo=0.10   # is_toxic confidence gate for pseudo-labels
LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed); print("Device:",DEVICE)

In [ ]:
# ---- 3. Metric helpers ----
def gate_preds(prob,thr,gate=True):
    pred=(prob>=thr).astype(int)
    if gate: pred[pred[:,0]==0,1:]=0
    return pred
def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    Bv=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+Bv)/2.0,A,Bv
def tune_thresholds(y,prob):
    grid=np.arange(0.10,0.90,0.01); thr=np.full(N_LABELS,0.5)
    bt,bs=0.5,-1
    for t in grid:
        s=f1_score(y[:,0],(prob[:,0]>=t).astype(int),average="macro",zero_division=0)
        if s>bs: bs,bt=s,t
    thr[0]=bt
    for i in range(1,N_LABELS):
        bt,bs=0.5,-1
        for t in grid:
            s=f1_score(y[:,i],(prob[:,i]>=t).astype(int),average="binary",zero_division=0)
            if s>bs: bs,bt=s,t
        thr[i]=bt if bs>0 else 0.5
    return thr

In [ ]:
# ---- 4. Load data ----
def find_data_dir():
    for c in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026","/kaggle/input/bengali-nlp-safety-challenge-2026"]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    hits=glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits: return os.path.dirname(sorted(hits,key=len)[0])
    for c in ["./data","."]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    raise FileNotFoundError("Attach the competition dataset.")
DATA_DIR=find_data_dir(); print("Data dir:",DATA_DIR)
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv")); test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
try: val_df=pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
except FileNotFoundError: val_df=None
def pick(df,pref,kw):
    for p in pref:
        if p in df.columns: return p
    for c in df.columns:
        if any(k in c.lower() for k in kw): return c
    o=[c for c in df.columns if df[c].dtype==object]; return max(o,key=lambda c: df[c].astype(str).str.len().mean())
TEXT_COL=pick(train_df,["text"],["text","comment"]); ID_COL=pick(test_df,["ID","id"],["id"])
full_df=pd.concat([train_df,val_df],ignore_index=True) if (val_df is not None and set(LABELS).issubset(val_df.columns)) else train_df.copy()
full_df=full_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
for l in LABELS: full_df[l]=full_df[l].astype(int)
print("Full",full_df.shape,"Test",test_df.shape)

In [ ]:
# ---- 5. Cleaning ----
def clean_text(t):
    if not isinstance(t,str): return ""
    t=re.sub(r"http\S+|www\.\S+","",t); t=re.sub(r"<.*?>","",t); t=re.sub(r"@\w+","",t)
    t=re.sub(r"\s+"," ",t).strip(); return t.replace("#","")
full_df["clean"]=full_df[TEXT_COL].apply(clean_text); test_df["clean"]=test_df[TEXT_COL].apply(clean_text)
print(full_df[["clean"]].head(2).to_string())

In [ ]:
# ---- 6. Dataset, Model, pos_weight ----
class DS(Dataset):
    def __init__(s,texts,tok,labels=None): s.t=list(texts); s.tok=tok; s.y=labels
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=s.tok(str(s.t[i]),add_special_tokens=True,max_length=CFG.max_len,padding="max_length",
                truncation=True,return_token_type_ids=False,return_attention_mask=True,return_tensors="pt")
        it={"input_ids":e["input_ids"].flatten(),"attention_mask":e["attention_mask"].flatten()}
        if s.y is not None: it["labels"]=torch.tensor(s.y[i],dtype=torch.float)
        return it
class BertBiLSTM(nn.Module):
    def __init__(s,name,n_out=7):
        super().__init__()
        s.bert=AutoModel.from_pretrained(name)
        s.lstm=nn.LSTM(s.bert.config.hidden_size,CFG.lstm_hidden,num_layers=2,batch_first=True,bidirectional=True,dropout=CFG.dropout)
        s.dropout=nn.Dropout(CFG.dropout); s.fc=nn.Linear(CFG.lstm_hidden*2,n_out)
    def forward(s,input_ids,attention_mask):
        seq=s.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        lo,_=s.lstm(seq); m=attention_mask.unsqueeze(-1).float()
        pooled=(lo*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return s.fc(s.dropout(pooled))
def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(neg/np.clip(pos,1,None),1.0,CFG.pos_weight_clip)
@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_fp16): lg=model(ids,mask)
        out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out,0)
print("model ready")

In [ ]:
# ---- 7. Train one fold (optionally with extra pseudo rows appended to TRAIN only) ----
def train_fold(mc,tok,trt,trY,vat,vaY,tet,fold,extra_t=None,extra_y=None):
    seed_everything(CFG.seed+fold)
    trt=list(trt); trY=np.asarray(trY,dtype=float)
    if extra_t is not None and len(extra_t):
        trt=trt+list(extra_t); trY=np.vstack([trY,np.asarray(extra_y,dtype=float)])
    model=BertBiLSTM(mc["name"],N_LABELS).to(DEVICE)
    trl=DataLoader(DS(trt,tok,trY),batch_size=mc["batch"],shuffle=True,num_workers=0,pin_memory=True)
    vl=DataLoader(DS(vat,tok,vaY),batch_size=64,shuffle=False,num_workers=0)
    el=DataLoader(DS(tet,tok),batch_size=64,shuffle=False,num_workers=0)
    pw=torch.tensor(pos_weight_from(trY),dtype=torch.float,device=DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    steps=max(1,(len(trl)//mc["accum"]))*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(steps*CFG.warmup_ratio),steps)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG.use_fp16); best,bv,bt=-1,None,None
    for ep in range(mc["epochs"]):
        model.train(); opt.zero_grad()
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE); y=b["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG.use_fp16): loss=crit(model(ids,mask),y)/mc["accum"]
            scaler.scale(loss).backward()
            if (st+1)%mc["accum"]==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        vp=predict(model,vl); th=tune_thresholds(vaY,vp); c,_,_=combined_score(vaY,gate_preds(vp,th,CFG.GATE))
        if c>=best: best=c; bv=vp; bt=predict(model,el)
    del model; gc.collect(); torch.cuda.empty_cache()
    return bv,bt

In [ ]:
# ---- 8. Ensemble runner (base+large), OOF-weighted blend ----
from sklearn.model_selection import StratifiedKFold
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold; HAVE_ITER=True
except Exception: HAVE_ITER=False
X=full_df["clean"].values; Y=full_df[LABELS].values.astype(float); test_texts=test_df["clean"].values
def make_folds(nf):
    if HAVE_ITER: return list(MultilabelStratifiedKFold(n_splits=nf,shuffle=True,random_state=42).split(X,Y.astype(int)))
    return list(StratifiedKFold(n_splits=nf,shuffle=True,random_state=42).split(X,Y[:,0].astype(int)))

def run_ensemble(extra_t=None, extra_y=None, tag=""):
    oofs=[]; tests=[]
    for mc in MODELS:
        print(f"  [{tag}] {mc['name'].split('/')[-1]} ({mc['folds']}f)")
        tok=AutoTokenizer.from_pretrained(mc["name"])
        oof=np.zeros((len(full_df),N_LABELS)); tp=np.zeros((len(test_df),N_LABELS))
        for fold,(tr,va) in enumerate(make_folds(mc["folds"])):
            vp,tt=train_fold(mc,tok,X[tr],Y[tr],X[va],Y[va],test_texts,fold,extra_t,extra_y)
            oof[va]=vp; tp+=tt/mc["folds"]
        oofs.append(oof); tests.append(tp)
    # OOF weight search
    best=(-1,0.5)
    for w in np.round(np.arange(0,1.01,0.05),2):
        ob=w*oofs[0]+(1-w)*oofs[1]; th=tune_thresholds(Y,ob)
        c,_,_=combined_score(Y,gate_preds(ob,th,CFG.GATE))
        if c>best[0]: best=(c,w)
    w=best[1]
    return w*oofs[0]+(1-w)*oofs[1], w*tests[0]+(1-w)*tests[1], w

In [ ]:
# ---- 9. ROUND 1 (real data) ----
print("########## ROUND 1 (train only) ##########")
oof1, test1, w1 = run_ensemble(tag="R1")
thr = tune_thresholds(Y, oof1)   # thresholds from REAL OOF (used for both rounds)
c1,A1,B1 = combined_score(Y, gate_preds(oof1,thr,CFG.GATE))
print(f"R1 blend w(base)={w1} OOF Combined={c1:.4f} (A={A1:.3f} B={B1:.3f})")

# ---- Build confident pseudo-labels from test ----
test1_hard = gate_preds(test1, thr, CFG.GATE)
conf = (test1[:,0]>=CFG.conf_hi) | (test1[:,0]<=CFG.conf_lo)   # confident on is_toxic
print(f"Confident test rows for pseudo-labeling: {int(conf.sum())}/{len(test_df)}")
pseudo_t = test_texts[conf]; pseudo_y = test1_hard[conf].astype(float)

In [ ]:
# ---- 10. ROUND 2 (train + confident pseudo) ----
print("########## ROUND 2 (train + pseudo) ##########")
oof2, test2, w2 = run_ensemble(extra_t=pseudo_t, extra_y=pseudo_y, tag="R2")
# final = average of the two rounds' test predictions (stability)
test_final = 0.5*test1 + 0.5*test2
# report R2 OOF (note: OOF here is on real rows, model also saw pseudo -> mildly optimistic)
c2,A2,B2 = combined_score(Y, gate_preds(oof2,thr,CFG.GATE))
print(f"R2 OOF Combined={c2:.4f} (A={A2:.3f} B={B2:.3f})")
np.save("test_final.npy",test_final)
# choose the better round on OOF for final thresholds/preds source
if c2>=c1:
    print("R2 >= R1 : using averaged test preds"); use=test_final; src="avg(R1,R2)"
else:
    print("R1 better: using R1 test preds"); use=test1; src="R1"
print("Final source:",src)

In [ ]:
# ---- 11. Submission (test.csv IDs) ----
final=gate_preds(use,thr,CFG.GATE)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].values})
for i,l in enumerate(LABELS): sub[l]=final[:,i]
sub.to_csv("submission.csv",index=False)
print("="*54)
print(f"Best OOF: R1={c1:.4f} R2={c2:.4f}  -> expected leaderboard ~ max of these")
print("submission.csv",sub.shape,"| IDs from official test.csv"); print(sub.head())